In [2]:
import numpy as np
import pandas as pd
import sys
import csv

# Increase the CSV field size limit to handle potentially very long strings
csv.field_size_limit(sys.maxsize)

data1 = pd.read_csv("emails.csv")
data2 = pd.read_csv("enron_spam_data.csv")
data3 = pd.read_csv("processed_data.csv", engine='python', on_bad_lines='skip')

In [3]:
print(data1.head())
print(data2.head())
print(data3.head())

                                                text  spam
0  Subject: naturally irresistible your corporate...     1
1  Subject: the stock trading gunslinger  fanny i...     1
2  Subject: unbelievable new homes made easy  im ...     1
3  Subject: 4 color printing special  request add...     1
4  Subject: do not have money , get software cds ...     1
   Unnamed: 0                       Subject  \
0           0  christmas tree farm pictures   
1           1      vastar resources , inc .   
2           2  calpine daily gas nomination   
3           3                    re : issue   
4           4     meter 7268 nov allocation   

                                             Message Spam/Ham        Date  
0                                                NaN      ham  1999-12-10  
1  gary , production from the high island larger ...      ham  1999-12-13  
2             - calpine daily gas nomination 1 . doc      ham  1999-12-14  
3  fyi - see note below - already done .\nstella\...      h

In [4]:
print(data1.describe())
print(data2.describe())
print(data3.describe())

              spam
count  5728.000000
mean      0.238827
std       0.426404
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000
         Unnamed: 0
count  33716.000000
mean   16857.500000
std     9733.115174
min        0.000000
25%     8428.750000
50%    16857.500000
75%    25286.250000
max    33715.000000
              label
count  75419.000000
mean       0.665602
std        0.471783
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000


### Spam/Ham Distribution in `data1`

In [5]:
print(data1['spam'].value_counts())
print(data1['spam'].value_counts(normalize=True) * 100)

spam
0    4360
1    1368
Name: count, dtype: int64
spam
0    76.117318
1    23.882682
Name: proportion, dtype: float64


### Spam/Ham Distribution in `data2`

In [6]:
print(data2['Spam/Ham'].value_counts())
print(data2['Spam/Ham'].value_counts(normalize=True) * 100)

Spam/Ham
spam    17171
ham     16545
Name: count, dtype: int64
Spam/Ham
spam    50.928343
ham     49.071657
Name: proportion, dtype: float64


### Spam/Ham Distribution in `data3`

In [7]:
print(data3['label'].value_counts())
print(data3['label'].value_counts(normalize=True) * 100)

label
1    50199
0    25220
Name: count, dtype: int64
label
1    66.560151
0    33.439849
Name: proportion, dtype: float64


### Combining Datasets

In [8]:
# Standardize data1: select 'text' and 'spam' columns
data1_processed = data1[['text', 'spam']].copy()

In [9]:
# Standardize data2: rename 'Message' to 'text', 'Spam/Ham' to 'spam', and convert labels
data2_processed = data2.rename(columns={'Message': 'text', 'Spam/Ham': 'spam'})[['text', 'spam']].copy()
data2_processed['spam'] = data2_processed['spam'].map({'spam': 1, 'ham': 0}).astype(int)

In [10]:
# Standardize data3: rename 'message' to 'text', 'label' to 'spam'
data3_processed = data3.rename(columns={'message': 'text', 'label': 'spam'})[['text', 'spam']].copy()

In [11]:
# Concatenate all processed dataframes
data_combined = pd.concat([data1_processed, data2_processed, data3_processed], ignore_index=True)

print("Combined DataFrame Head:")
print(data_combined.head())
print("\nCombined DataFrame Info:")
print(data_combined.info())

Combined DataFrame Head:
                                                text  spam
0  Subject: naturally irresistible your corporate...     1
1  Subject: the stock trading gunslinger  fanny i...     1
2  Subject: unbelievable new homes made easy  im ...     1
3  Subject: 4 color printing special  request add...     1
4  Subject: do not have money , get software cds ...     1

Combined DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114863 entries, 0 to 114862
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    113324 non-null  object
 1   spam    114863 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.8+ MB
None


### Spam/Ham Distribution in Combined Dataset

In [12]:
print(data_combined['spam'].value_counts())
print(data_combined['spam'].value_counts(normalize=True) * 100)

spam
1    68738
0    46125
Name: count, dtype: int64
spam
1    59.843466
0    40.156534
Name: proportion, dtype: float64


### Splitting the Dataset

In [13]:
from sklearn.model_selection import train_test_split

# Drop rows where 'text' is NaN before splitting
data_combined_cleaned = data_combined.dropna(subset=['text'])

X = data_combined_cleaned['text']
y = data_combined_cleaned['spam']

# Split 80% train, 20% (test+eval)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Split X_temp and y_temp into 10% test and 10% eval
X_test, X_eval, y_test, y_eval = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train set size: {len(X_train)} samples")
print(f"Test set size: {len(X_test)} samples")
print(f"Evaluation set size: {len(X_eval)} samples")

Train set size: 90659 samples
Test set size: 11332 samples
Evaluation set size: 11333 samples


### TF-IDF Vectorization

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

# Fit on training data and transform all sets
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
X_eval_tfidf = tfidf_vectorizer.transform(X_eval)

print(f"TF-IDF transformed training data shape: {X_train_tfidf.shape}")
print(f"TF-IDF transformed testing data shape: {X_test_tfidf.shape}")
print(f"TF-IDF transformed evaluation data shape: {X_eval_tfidf.shape}")

TF-IDF transformed training data shape: (90659, 5000)
TF-IDF transformed testing data shape: (11332, 5000)
TF-IDF transformed evaluation data shape: (11333, 5000)


### Random Forest Model Development and Evaluation

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize Random Forest Classifier
random_forest_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Train the model
print("Training Random Forest model...")
random_forest_model.fit(X_train_tfidf, y_train)
print("Training complete.")

# Make predictions on the test set
y_pred = random_forest_model.predict(X_test_tfidf)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"\nModel Accuracy on Test Set: {accuracy:.4f}")
print("\nClassification Report on Test Set:")
print(report)

Training Random Forest model...
Training complete.

Model Accuracy on Test Set: 0.9921

Classification Report on Test Set:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4607
           1       0.99      0.99      0.99      6725

    accuracy                           0.99     11332
   macro avg       0.99      0.99      0.99     11332
weighted avg       0.99      0.99      0.99     11332



### Prediction Function for New Emails

In [16]:
def predict_spam_ham(email_text):
    """
    Predicts whether an email is spam or ham using the trained Random Forest model.

    Args:
        email_text (str): The text content of the email.

    Returns:
        str: 'spam' if the email is predicted as spam, 'ham' otherwise.
    """
    # Transform the email text using the fitted TF-IDF vectorizer
    email_tfidf = tfidf_vectorizer.transform([email_text])

    # Predict using the trained Random Forest model
    prediction = random_forest_model.predict(email_tfidf)

    # Return 'spam' or 'ham' based on the prediction
    if prediction[0] == 1:
        return 'spam'
    else:
        return 'ham'

# --- Demonstration ---
# Example spam email
spam_email_example = "Subject: Claim your prize now! You have won a lottery of $1,000,000! Click here to claim."
print(f"' {spam_email_example[:50]}...' is predicted as: {predict_spam_ham(spam_email_example)}")

# Example ham email
ham_email_example = "Subject: Meeting Reminder Hi team, just a friendly reminder about our meeting tomorrow at 10 AM. See you there!"
print(f"' {ham_email_example[:50]}...' is predicted as: {predict_spam_ham(ham_email_example)}")

# Another example spam email
spam_email_example_2 = "Subject: Urgent! Your account will be closed. Verify your details here: http://malicious.link"
print(f"' {spam_email_example_2[:50]}...' is predicted as: {predict_spam_ham(spam_email_example_2)}")

' Subject: Claim your prize now! You have won a lott...' is predicted as: spam
' Subject: Meeting Reminder Hi team, just a friendly...' is predicted as: ham
' Subject: Urgent! Your account will be closed. Veri...' is predicted as: spam



### XGBoost Model Development and Evaluation

In [17]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

# Initialize XGBoost Classifier
# Using use_label_encoder=False and eval_metric for modern XGBoost versions
xgboost_model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss', n_jobs=-1)

# Train the model
print("Training XGBoost model...")
xgboost_model.fit(X_train_tfidf, y_train)
print("Training complete.")

# Make predictions on the test set
y_pred_xgb = xgboost_model.predict(X_test_tfidf)

# Evaluate the model
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
report_xgb = classification_report(y_test, y_pred_xgb)

print(f"\nModel Accuracy on Test Set (XGBoost): {accuracy_xgb:.4f}")
print("\nClassification Report on Test Set (XGBoost):")
print(report_xgb)

Training XGBoost model...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:54:11] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training complete.

Model Accuracy on Test Set (XGBoost): 0.9912

Classification Report on Test Set (XGBoost):
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4607
           1       0.99      0.99      0.99      6725

    accuracy                           0.99     11332
   macro avg       0.99      0.99      0.99     11332
weighted avg       0.99      0.99      0.99     11332



### Prediction Function for New Emails (XGBoost Model)

In [23]:
def predict_spam_ham_xgb(email_text):
    """
    Predicts whether an email is spam or ham using the trained XGBoost model.

    Args:
        email_text (str): The text content of the email.

    Returns:
        str: 'spam' if the email is predicted as spam, 'ham' otherwise.
    """
    # Transform the email text using the fitted TF-IDF vectorizer
    email_tfidf = tfidf_vectorizer.transform([email_text])

    # Predict using the trained XGBoost model
    prediction = xgboost_model.predict(email_tfidf)

    # Return 'spam' or 'ham' based on the prediction
    if prediction[0] == 1:
        return 'spam'
    else:
        return 'ham'

# --- Demonstration ---
# Example spam email
spam_email_example = "Subject: Claim your prize now! You have won a lottery of $1,000,000! Click here to claim."
print(f"' {spam_email_example[:50]}...' is predicted by XGBoost as: {predict_spam_ham_xgb(spam_email_example)}")

# Example ham email
ham_email_example = "Subject: Meeting Reminder Hi team, just a friendly reminder about our meeting tomorrow at 10 AM. See you there!"
print(f"' {ham_email_example[:50]}...' is predicted by XGBoost as: {predict_spam_ham_xgb(ham_email_example)}")

# Another example spam email
spam_email_example_2 = "Subject: Urgent! Your account will be closed. Verify your details here: http://malicious.link"
print(f"' {spam_email_example_2[:50]}...' is predicted by XGBoost as: {predict_spam_ham_xgb(spam_email_example_2)}")

' Subject: Claim your prize now! You have won a lott...' is predicted by XGBoost as: spam
' Subject: Meeting Reminder Hi team, just a friendly...' is predicted by XGBoost as: ham
' Subject: Urgent! Your account will be closed. Veri...' is predicted by XGBoost as: spam


### BERT-uncased Model Development

In [21]:
# Install necessary libraries
!pip install transformers torch accelerate -q

In [26]:
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW # Corrected import path for AdamW
from sklearn.metrics import accuracy_score, classification_report
from tqdm.notebook import tqdm

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"Using device: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using device: cpu


#### Tokenizing Data for BERT

In [ ]:
# Tokenize the datasets
max_length = 512 # BERT's maximum sequence length

# Encode training data
encodings_train = tokenizer(list(X_train), truncation=True, padding=True, max_length=max_length)

# Encode test data
encodings_test = tokenizer(list(X_test), truncation=True, padding=True, max_length=max_length)

# Encode evaluation data
encodings_eval = tokenizer(list(X_eval), truncation=True, padding=True, max_length=max_length)

print("Data tokenization complete.")
print(f"Example of tokenized input_ids (first 10 from training set): {encodings_train['input_ids'][0][:10]}")

#### Creating Custom Dataset and DataLoaders

In [ ]:
class SpamHamDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Create Dataset objects
train_dataset = SpamHamDataset(encodings_train, list(y_train))
test_dataset = SpamHamDataset(encodings_test, list(y_test))
eval_dataset = SpamHamDataset(encodings_eval, list(y_eval))

# Create DataLoaders
batch_size = 16 # Adjust based on GPU memory
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
eval_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)

print("Custom datasets and DataLoaders created.")

#### Training the BERT Model

In [ ]:
epochs = 3 # Number of training epochs
learning_rate = 5e-5 # Learning rate

optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# Training loop
model.train() # Set model to training mode

print("Starting BERT model training...")
for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Clip gradients to prevent exploding gradients
        optimizer.step()
        scheduler.step()

print("BERT model training complete.")

#### Evaluating the BERT Model

In [ ]:
model.eval() # Set model to evaluation mode

predictions = []
true_labels = []

print("Evaluating BERT model on the test set...")
with torch.no_grad(): # Disable gradient calculations for evaluation
    for batch in tqdm(test_loader, desc="Evaluating"): # Use test_loader for evaluation
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1).flatten()
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

# Calculate accuracy and print classification report
accuracy_bert = accuracy_score(true_labels, predictions)
report_bert = classification_report(true_labels, predictions)

print(f"\nBERT Model Accuracy on Test Set: {accuracy_bert:.4f}")
print("\nClassification Report on Test Set (BERT):")
print(report_bert)

### Saving Datasets

In [19]:
# Save the combined and cleaned DataFrame to a CSV file
data_combined_cleaned.to_csv('data_combined_cleaned.csv', index=False)
print("Saved 'data_combined_cleaned.csv'")

Saved 'data_combined_cleaned.csv'


In [20]:
from scipy.sparse import save_npz

# Save TF-IDF transformed sparse matrices
save_npz('X_train_tfidf.npz', X_train_tfidf)
save_npz('X_test_tfidf.npz', X_test_tfidf)
save_npz('X_eval_tfidf.npz', X_eval_tfidf)

print("Saved 'X_train_tfidf.npz', 'X_test_tfidf.npz', and 'X_eval_tfidf.npz'")
print("You can find these files in the Colab file browser (folder icon on the left) and download them.")

Saved 'X_train_tfidf.npz', 'X_test_tfidf.npz', and 'X_eval_tfidf.npz'
You can find these files in the Colab file browser (folder icon on the left) and download them.
